### Preprocessing 
#### Finetune prompt

In [1]:
import json
import os
from docx import Document
from openai import AzureOpenAI


In [2]:
client = AzureOpenAI(
         azure_endpoint = "azure_endpoint", 
         api_key = "api_key",  
         api_version = "api_version"
         )

In [3]:
def get_completion(messages):
    """ GET completion from openai api"""
    response = client.chat.completions.create(
        model = "model", 
        messages = messages,
        max_tokens = 6000,
        temperature = 0.7,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
)
    return response

In [4]:
def extract_text_from_docx(file_path):
    doc = Document(file_path)
    full_text = []
    for paragraph in doc.paragraphs:
        full_text.append(paragraph.text)
    return '\n'.join(full_text)

In [5]:
def clue_prompt(dialogue, topics, clue_instruction):
    prompt = (
        f"{clue_instruction}\n\n"
        f"Dialogue:\n{dialogue}\n\n"
        f"Topics:\n{topics}\n\n"
        f"Clues: "
    )
    
    messages = [
        {
            "role": "system",
            "content": """
            You are a linguistic expert with extensive experience analyzing patient-doctor dialogues. 
            These patients are diagnosed with heart failure. You are trying to understand patient's perception of the intensity of heart failure medications.
            Your task is to extract key clues (limit to 200 words) diectly from original dialogues supporting each given identified topic.
            
            Clues must:
            - Be direct quotes from the dialogue (no summarization or interpretation).
            - Be brief but contextually complete.
            - Highlight key phrases, contextual information, emotional tones, or symptoms related to the topic.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return messages

In [6]:
def reasoning_prompt(clues, topics, reasoning_instruction):
    prompt = (
        f"{reasoning_instruction}\n\n"
        f"Clues:\n{clues}\n\n"
        f"Topics:\n{topics}\n\n"
        f"Reasonings: "
        )
    
    messages = [
        {
            "role": "system",
            "content": """
            You are a linguistic expert tasked with analyzing patient-doctor dialogues. 
            These patients are diagnosed with heart failure.
            You are trying to understand patient's perception of the intensity of heart failure medications.
            Your goal is to provide a clear and concise reasoning process (limit to 150 words) based on provided clues to explain each corresponding identified topic.

            Ensure your reasoning:
            - Links the clues directly to the topic.
            - Explains the logical connection between the clues and topic.
            - Avoids adding external context or information not present in the clues.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    return messages

In [7]:
def evaluation_prompt_batch(json_file_paths):
    prompt = ""

    for i, file_path in enumerate(json_file_paths):
        with open(file_path, "r") as file:
            data = json.load(file)
            clues = data.get("clues", "N/A")
            reasoning = data.get("reasoning", "N/A")
            topics = data.get("topics", "N/A")
        
        # Append each dialogue's evaluation content to the prompt
        prompt += (
            f"### Clues-Reasoning-Topic Pair {i + 1} ###\n\n"
            f"**Clues:** {clues}\n\n"
            f"**Reasoning:** {reasoning}\n\n"
            f"**Topics:** {topics}\n\n"
            "### Evaluation Task ###\n"
            "For the above Clues-Reasoning-Topic pair:\n"
            "1. **Clue Quality:** Evaluate the clues based on the following:\n"
            "   - How relevant and accurate are the clues in supporting the topic(s)?\n"
            "   - Are the clues complete (include all key information) and free of irrelevant details?\n"
            "   - Do the clues contain context or are they missing critical information from the dialogue?\n\n"
            "2. **Reasoning Quality:** Assess the reasoning based on the following:\n"
            "   - Does the reasoning logically connect the clues to the topic(s)?\n"
            "   - Are there any gaps or missing logic in the reasoning process?\n"
            "   - Is the reasoning concise and free of unnecessary content?\n\n"
        )
    
    # Add the aggregate feedback section
    prompt += (
        "### Aggregate Feedback Task ###\n"
        "Based on your evaluation of all the Clues-Reasoning-Topic pairs, provide:\n\n"
        "**Common Issues:**\n"
        "- **Clue Generation:** Identify recurring problems in the generated clues (e.g., missing context, irrelevant clues).\n"
        "- **Reasoning Generation:** Highlight frequent issues in reasoning (e.g., logical gaps, weak connections between clues and topics).\n\n"
        "**Suggestions for Improvement:**\n"
        "- **Clue Prompt:** Propose specific improvements to the clue generation prompt.\n"
        "- **Reasoning Prompt:** Recommend actionable enhancements to the reasoning generation prompt.\n\n"
    )
    
    messages = [
        {
            "role": "system",
            "content": """
            You are an evaluation expert tasked with analyzing a BATCH of patient-doctor dialogue clue-reasoning-topic pairs. 
            These patients are diagnosed with heart failure, and the focus is on understanding their perception of the intensity of heart failure medications. 
            
            Your tasks are as follows:\n
            1. Evaluate each Clues-Reasoning-Topic pair for Clue Quality and Reasoning Quality.\n
            2. Provide feedback on both the relevance and completeness of the clues, and the logical coherence of the reasoning.\n
            3. Identify common issues across all pairs in clue and reasoning generation.\n
            4. Suggest improvements to the clue and reasoning prompts based on recurring patterns of errors.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    return messages

In [8]:
def optimization_prompt(clue_prompt, reasoning_prompt, feedback):
    prompt = (
        "You are tasked with improving two prompts based on provided feedback.\n\n"
        "### Task Description ###\n"
        "Given the following feedback, improve both the **Clue Prompt** and the **Reasoning Prompt** simultaneously to address the issues and suggestions provided:\n\n"
        "1. The **Clue Prompt** should:\n"
        "   - Guide the user/system to extract relevant, precise, and contextually complete clues directly from the dialogue.\n"
        "   - Ensure the clues are accurate quotes, avoid irrelevant or incomplete clues, and incorporate missing elements identified in the feedback.\n"
        "   - Focus on ensuring clarity and usability of the prompt.\n\n"
        "2. The **Reasoning Prompt** should:\n"
        "   - Guide the user/system to logically and effectively connect the clues to the identified topics.\n"
        "   - Ensure the reasoning structure is clear, addresses logical gaps, and builds a strong link between the clues and topics.\n"
        "   - Incorporate improvements to reasoning clarity and structure as per the feedback.\n\n"
        "### Provided Inputs ###\n"
        f"**Feedback:**\n{feedback}\n\n"
        f"**Current Clue Prompt:**\n{clue_prompt}\n\n"
        f"**Current Reasoning Prompt:**\n{reasoning_prompt}\n\n"
        "### Output Instructions ###\n"
        "You MUST provide your improved prompts formatted as follows:\n"
        "- For the clue prompt: `<IMPROVED_CLUE_PROMPT> your improved clue prompt text </IMPROVED_CLUE_PROMPT>`\n"
        "- For the reasoning prompt: `<IMPROVED_REASONING_PROMPT> your improved reasoning prompt text </IMPROVED_REASONING_PROMPT>`\n\n"
        "The text provided between these tags will directly replace the current prompts, so ensure your improvements are complete, clear, and directly address the feedback provided.\n\n"
    )

    messages = [
        {
            "role": "system",
            "content": """
            You are part of an optimization system that improves text. You will be asked to creatively and critically improve the clue prompt and reasoing prompt (instructions). 
            You will receive some feedback, and use the feedback to improve both clue and reasoning prompts simultaneously. The feedback may be noisy, identify what is important and what is correct. 
            Pay attention to the role description of the clue and reasoning prompts (instructions), and the context in which it is used. 
            """
    
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    return messages

In [9]:
def complete_workflow(docx_file_paths, topics_list, clue_instruction, reasoning_instruction, json_output_dir, iteration, price_per_1000_input_tokens=0.00015, price_per_1000_output_tokens=0.0006):
    json_file_paths = []

    for i, file_path in enumerate(docx_file_paths):
        try:
            # Step 1: Extract dialogue from `.docx`
            dialogue = extract_text_from_docx(file_path)

            # Get the corresponding topics
            topics = topics_list[i]

            # Step 2: Generate clues using `clue_prompt`
            clue_messages = clue_prompt(dialogue, topics, clue_instruction)
            clue_response = get_completion(clue_messages)
            clues = clue_response.choices[0].message.content.strip()
        
            # Step 3: Generate reasoning using `reasoning_prompt`
            reasoning_messages = reasoning_prompt(clues, topics, reasoning_instruction)
            reasoning_response = get_completion(reasoning_messages)
            reasoning = reasoning_response.choices[0].message.content.strip()

            # Save intermediate results to JSON
            json_file_path = f"{json_output_dir}/{file_path.split('/')[-1].replace('.docx', f'_iteration_{iteration + 1}.json')}"
            json_file_paths.append(json_file_path)
            with open(json_file_path, "w") as json_file:
                json.dump({
                    "dialogue": file_path,
                    "clues": clues,
                    "reasoning": reasoning,
                    "topics": topics
                }, json_file, indent=4)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    # Step 4: Evaluate batches using `evaluation_prompt_batch`
    evaluation_messages = evaluation_prompt_batch(json_file_paths)
    evaluation_response = get_completion(evaluation_messages)

    aggregate_feedback_marker = "Aggregate Feedback"
    agg_feedback = ""
    if aggregate_feedback_marker in evaluation_response.choices[0].message.content:
        agg_feedback = evaluation_response.choices[0].message.content.split(aggregate_feedback_marker, 1)[1]   
    
    # Step 5: Optimize prompts using `optimization_prompt`
    optimized_messages = optimization_prompt(clue_instruction, reasoning_instruction, agg_feedback)
    optimization_response = get_completion(optimized_messages)
    optimized_clue_prompt = optimization_response.choices[0].message.content.split("<IMPROVED_CLUE_PROMPT>")[1].split("</IMPROVED_CLUE_PROMPT>")[0].strip()
    optimized_reasoning_prompt = optimization_response.choices[0].message.content.split("<IMPROVED_REASONING_PROMPT>")[1].split("</IMPROVED_REASONING_PROMPT>")[0].strip()

    # Output the results for this iteration
    return {
        "feedback": agg_feedback,
        "optimized_clue_prompt": optimized_clue_prompt,
        "optimized_reasoning_prompt": optimized_reasoning_prompt
    }

In [10]:
def iterative_workflow(
    docx_file_paths, topics_list, initial_clue_instruction, initial_reasoning_instruction, json_output_dir, num_iterations, price_per_1000_input_tokens=0.00015, price_per_1000_output_tokens=0.0006
):
    clue_instruction = initial_clue_instruction
    reasoning_instruction = initial_reasoning_instruction
    final_results = {}

    for iteration in range(num_iterations):
        print(f"Running iteration {iteration + 1}/{num_iterations}...")
        iteration_json_output_dir = f"{json_output_dir}/iteration_{iteration + 1}"
        os.makedirs(iteration_json_output_dir, exist_ok=True)
        
        # Step 1: Run the complete workflow for this iteration
        results = complete_workflow(docx_file_paths, topics_list, clue_instruction, reasoning_instruction, iteration_json_output_dir, iteration, price_per_1000_input_tokens, price_per_1000_output_tokens)

        # Update the prompts for the next iteration
        clue_instruction = results["optimized_clue_prompt"]
        reasoning_instruction = results["optimized_reasoning_prompt"]

        final_results[f"Iteration {iteration + 1}"] = {
            "clue_prompt": clue_instruction,
            "reasoning_prompt": reasoning_instruction,
            "feedback": results["feedback"]
        }

        print(f"\nIteration {iteration + 1} Feedback:")
        print(results["feedback"])
        print(f"\nOptimized Clue Prompt (Iteration {iteration + 1}):")
        print(clue_instruction)
        print(f"\nOptimized Reasoning Prompt (Iteration {iteration + 1}):")
        print(reasoning_instruction)

    return final_results

In [ ]:
# usage example 
docx_file_paths = [
    "file_path_1.docx",
    "file_path_2.docx",
    "file_path_3.docx"
]
    
topics_list = [
    "Topic 1, Topic 2, Topic 3",
    "Topic 1, Topic 2, Topic 3, Topic 4",
    "Topic 1, Topic 2, Topic 3",
]
    
initial_clue_instruction = "List clues (i.e. key phrases, contextual information, semantic and emotional tones, temporal information) in the following interviews that support each given identified topic."

initial_reasoning_instruction = "Based on the given clues, generate the reasoning process that supports the identified topics.\n\n"

json_output_dir = "output_file_path"

num_iterations = 3

final_results = iterative_workflow(
        docx_file_paths, topics_list, initial_clue_instruction, initial_reasoning_instruction, json_output_dir, num_iterations, price_per_1000_input_tokens=0.00015, price_per_1000_output_tokens=0.0006
    )

# Print final optimized prompts and feedback
final_iteration_results = final_results[f"Iteration {num_iterations}"]

print("\nFinal Optimized Clue Prompt:")
print(final_iteration_results["clue_prompt"])

print("\nFinal Optimized Reasoning Prompt:")
print(final_iteration_results["reasoning_prompt"])

print("\nFeedback for Final Iteration:")
print(final_iteration_results["feedback"])

### Inference

In [ ]:
optimized_clue = "Obtained from iteration results"
optimized_reasoning = "Obtained from iteration results"

In [11]:
def topic_identification_prompt(dialogue, optimized_clue, optimized_reasoning):
    prompt = (
        "Your task is to identify ALL applicable topics for the given dialogue."
        "Topics should be about patient's perception of the intensity of heart failure medications"
        "Each topic should be concise, meaningful, and specific. Avoid combining distinct ideas or using vague terms.\n\n"
        "There may be multiple topics, so ensure you capture each distinct one.\n\n"
        f"Step 1 Extract CLUES: {optimized_clue}\n\n"
        f"Step 2 Generate REASONING: {optimized_reasoning}\n\n"
        "Step 3 Identify TOPICS: Based on the dialogue, clues, and reasoning, identify all applicable topics.\n\n"

        "### IMPORTANT REQUIREMENTS FOR IDENTIFIED TOPICS ###\n\n"
        "- **Clarity:** Use precise and specific language, avoiding vague or ambiguous terms such as 'perception' or 'impact' without emotional context.\n\n"
        "- **Emotional Context:** Clearly indicate the nature of any perceptions, emotions, or reactions (e.g., positive, negative) as they appear in the dialogue.\n\n"
        "- **Single Concept:** Ensure each topic represents one distinct idea, avoiding the merging of separate concepts.\n\n"
        "- **Relevance and Specificity:** Make topics meaningful, actionable, and directly related to the context of the dialogue.\n\n"
        
        "### Examples ###\n\n"
        "Good Topics:\n\n"
        "- **Burden from the number of medications** (highlights the number of medications taken as a significant burden)\n\n"
        "- **Problem in logistics** (reports issues related to obtaining medications)\n\n"
        "- **Impact from patient-doctor relationship** (discusses how the interaction between patient and doctor/healthcare system influences)\n\n"
        "- **Adverse drug effects** (report the patient's experience of side effects from medications)\n\n"
        "Bad Topics:\n\n"
        "- **Medication management support** (vague and unclear. It does not specify whether the patient received support, lacked support, or faced issues related to medication management.)\n\n"
        "- **Perceived medication burden** (vague and unclear. It does not provide sufficient information about whether or not the patient experienced a medication number burden)\n\n"
        "- **Emotional impact of medication side effects** (combines two distinct concepts into one)\n\n"
        "- **Medication adherence and management** (vague and unclear, combines two distinct concepts into one)\n\n"
        "Make sure each identified topic follows good topic examples and avoids bad topic examples.\n\n"
        
        "### Output Format ###\n\n"
        "For EACH identified topic, provide the following EXACTLY:\n\n"
        "Identify topic: [Insert topic here]\n\n"
        "Clues (max 200 words): [Insert clues here]\n\n"
        "Reasoning (max 150 words): [Insert reasoning here]\n\n"
        f"Dialogue: {dialogue}\n\n"
    )
    
    messages = [
        {
            "role": "system",
            "content": """You are a medical expert tasked with identifying topics of patient-doctor dialogues.
            These patients are diagnosed with heart failure. You are trying to understand patient's perception of the intensity of heart failure medications.
            
            Your task:
            - Identify **all applicable topics** for the given dialogue (there may be more than one).
            - For each identified topic, provide clues and reasoning to explain the connection.
            
            Clues must:
            - Be direct quotes from the dialogue (no summarization or interpretation).
            - Be brief but contextually complete.
            - Highlight key phrases, contextual information, emotional tones, or symptoms related to the topic.

            Reasoning must:
            - Links the clues directly to the topic.
            - Explains the logical connection between the clues and topic.
            - Avoids adding external context or information not present in the clues.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return messages

In [12]:
def process_multiple_dialogues(dialogue_file_paths, optimized_clue, optimized_reasoning, output_file_path):
    results = {}

    for file_path in dialogue_file_paths:
        try:
            dialogue = extract_text_from_docx(file_path)
            messages = topic_identification_prompt(dialogue, optimized_clue, optimized_reasoning)
            response = get_completion(messages)
            res = response.choices[0].message.content.strip()
            results[file_path] = res
            print(f"{file_path} has been processed.")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            results[file_path] = {"error": str(e)}

    with open(output_file_path, "w") as json_file:
        json.dump(results, json_file, indent=4)

    return results

In [ ]:
# usage example 
test_docx_files = [
    "test_file_path_1.docx",
    "test_file_path_2.docx",
    "test_file_path_3.docx"
]

output_file_path = "output_file_path/results.json"

topic_results = process_multiple_dialogues(test_docx_files, optimized_clue, optimized_reasoning, output_file_path)

#### Self-consistency

In [13]:
def process_multiple_dialogues(dialogue_file_paths, optimized_clue, optimized_reasoning, output_dir, n):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)  # Create output directory if it doesn't exist

    summary = {}

    for dialogue_file_path in dialogue_file_paths:
        dialogue_name = os.path.splitext(os.path.basename(dialogue_file_path))[0]
        output_file_path = os.path.join(output_dir, f"{dialogue_name}_results.json")

        results = {}

        try:
            dialogue = extract_text_from_docx(dialogue_file_path)

            for run in range(1, n+1):  
                messages = topic_identification_prompt(dialogue, optimized_clue, optimized_reasoning)
                response = get_completion(messages)  
                res = response.choices[0].message.content.strip()
                results[f"run_{run}"] = res
    
        except Exception as e:
            print(f"Error processing {dialogue_file_path}: {e}")
            results["error"] = str(e)

        with open(output_file_path, "w") as json_file:
            json.dump(results, json_file, indent=4)

        summary[dialogue_file_path] = output_file_path

    return summary

In [ ]:
# usage example
test_docx_files = [
    "test_file_path_1.docx",
    "test_file_path_2.docx",
    "test_file_path_3.docx"
]

output_dir = "output_dir"

summary = process_multiple_dialogues(test_docx_files, optimized_clue, optimized_reasoning, output_dir, n=3) # run 3 times for each dialogue

In [14]:
def common_topics(json_file_path):
    try:
        with open(json_file_path, "r") as file:
            outputs = json.load(file)
    except Exception as e:
        print(f"Error reading file {json_file_path}: {e}")
        return None

    prompt = (
        "You are analyzing topic identification outputs from multiple analyses of the same patient-doctor dialogue.\n\n"
        "### Your Goal ###\n\n"
        "Identify common topics across multiple outputs that appear in at least two outputs. For a topic to be considered common, it must:\n"
        "- Have similar meaning.\n"
        "- Be supported by similar extracted clues.\n"
        "- Have similar reasoning.\n\n"
        "**Important:**\n"
        "- Do NOT identify common topics within a single output.\n"
        "- Only compare across multiple outputs.\n\n"
        
        "### Instructions ###\n"
        "For each common topic:\n"
        "1. Select the best topic name from the outputs that represents the common topic.\n"
        "2. Aggregate all associated clues (without modification).\n"
        "3. Summarize the reasoning concisely.\n\n"

        "### Examples ###\n\n"
        "Good Topics:\n\n"
        "- **Burden from the number of medications** (highlights the number of medications taken as a significant burden)\n\n"
        "- **Problem in logistics** (reports issues related to obtaining medications)\n\n"
        "- **Impact from patient-doctor relationship** (discusses how the interaction between patient and doctor/healthcare system influences)\n\n"
        "- **Adverse drug effects** (report the patient's experience of side effects from medications)\n\n"
        "Bad Topics:\n\n"
        "- **Medication management support** (vague and unclear. It does not specify whether the patient received support, lacked support, or faced issues related to medication management.)\n\n"
        "- **Perceived medication burden** (vague and unclear. It does not provide sufficient information about whether or not the patient experienced a medication number burden)\n\n"
        "- **Emotional impact of medication side effects** (combines two distinct concepts into one)\n\n"
        "- **Medication adherence and management** (vague and unclear, combines two distinct concepts into one)\n\n"
        "Make sure each selected identified common topic follows good topic examples and avoids bad topic examples.\n\n"
        
        "### Output Format ###\n"
        "Provide your results in the following format for EACH common topic:\n\n"
        "Topic: [Insert best topic name]\n\n"
        "Clues (max 200 words): [Insert aggregated clues]\n\n"
        "Reasoning (max 150 words): [Insert summarized reasoning]\n\n"
        "If no common topics are found, respond with:\n"
        "'No common topics found.'\n\n"
    )

    for idx, content in enumerate(outputs.values(), start=1): 
        prompt += f"Output {idx}:\n{content}\n\n"
    
    messages = [
        {
            "role": "system",
            "content": "You are an advanced AI designed to analyze outputs from topic identification tasks. Your job is to identify and process common topics based on meaning, clues, and reasoning across multiple outputs."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return messages

In [15]:
def process_common_topics(summary, processed_results_dir, common_results_dir):
    os.makedirs(common_results_dir, exist_ok=True)

    common_results_summary = {}

    for dialogue_file_path, json_file_path in summary.items():
        try:
            dialogue_name = os.path.splitext(os.path.basename(dialogue_file_path))[0]
            common_file_path = os.path.join(common_results_dir, f"{dialogue_name}_common.json")

            messages = common_topics(json_file_path)
            response = get_completion(messages)  
            commons = response.choices[0].message.content.strip()

            with open(common_file_path, "w") as file:
                json.dump({"common_topics": commons}, file, indent=4)

            common_results_summary[dialogue_file_path] = common_file_path

        except Exception as e:
            print(f"Error consolidating topics for {dialogue_file_path}: {e}")

    return common_results_summary

In [ ]:
# usage example
processed_results_dir = "processed_results_dir"
common_results_dir = "common_results_dir"

common_results = process_common_topics(summary, processed_results_dir, common_results_dir)

#### Topic Cluster

In [16]:
def topic_cluster_explain_prompt(dataframe):
    topic_clue_reasoning_blocks = ""

    # Count the total number of original topics
    total_topics = len(dataframe)

    # Construct the topic blocks from the DataFrame
    for i, row in dataframe.iterrows():
        topic_clue_reasoning_blocks += (
            f"Topic {i + 1}: {row['Identify Topic']}\n"
            f"Clues: {row['Clues']}\n"
            f"Reasoning: {row['Reasoning']}\n\n"
        )
    
    prompt = (
        f"You are given {total_topics} original topics, along with their associated clues and reasoning:\n\n"
        "Your task is to analyze all the topics with the corresponding clues and reasoning. "
        "Identify topics with clues and reasonings that are similar, semantically related, or duplicates, and merge them into several distinct and non-overlapping consolidated topics.\n"
        "If some topics cannot be merged with others, leave them as standalone consolidated topics in the final output.\n"
        "Double-check that all original topics are accounted for in your final output.\n"
        "============\n"
        "### STRICT REQUIREMENTS FOR CONSOLIDATED TOPICS ###\n\n"
        "- **Single Concept:** Each consolidated topic must represent one distinct concept. Do not combine unrelated or separate ideas into a single topic.\n\n"
        "- **Clarity and Specificity:** Consolidated topics should be concise, meaningful, actionable, and specific.\n\n"
        "- **Semantic Similarity:** Only merge topics that are semantically related or synonymous, sharing the same underlying concept.\n\n"
        "- **Preserve Original Meaning:** Ensure the original meaning of each merged topic is maintained without introducing new interpretations.\n\n"
        "- **Standalone Topics:** Topics that cannot be semantically merged with others should remain as standalone consolidated topics.\n\n"
        "- **Complete Representation:** Confirm that all original topics ({total_topics}) are fully represented in the final consolidated output.\n\n"
        "### Output Format ###\n\n"
        "For EACH consolidated topic, provide the following:\n"
        "Consolidated topic: [Insert consolidated topic here]\n\n"
        "Original topics: [Insert original topic names here]\n\n"
        "Explanation (max 150 words): [Insert explanation here]\n\n"
        "============\n\n"
        f"{topic_clue_reasoning_blocks}\n\n"
        )
    
    messages = [
        {
            "role": "system",
            "content": """
            Your task is to analyze the provided list of topics, along with their associated clues and reasoning, to identify and merge topics that are either similar or duplicates. 
            Focus strictly on merging topics that are semantically related or synonymous, sharing the same core concept. 
            Under no circumstances should distinct topics be merged into one consolidated topic. 
            Topics that cannot be merged with others must remain as standalone consolidated topics.
            Double-check your output to ensure that all original topics are fully represented in the final output. Explicitly mention any topics that were not included in the consolidated topics.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    return messages

In [ ]:
# usage example
## df is the dataset that contains identified topics, extracted clues, generated reasoning for each dialogue
messages = topic_cluster_explain_prompt(df)
response = get_completion(messages)
cluster_explain_text = response.choices[0].message.content
print(cluster_explain_text)